# Data Loading

In [1]:
import pandas as pd
import re
import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [2]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
reviews_df.head()

,app_id,app_name,review_text,review_score,review_votes
0,10,Counter-Strike,Ruined my life.,1,0
1,10,Counter-Strike,This will be more of a ''my experience with th...,1,1
2,10,Counter-Strike,This game saved my virginity.,1,0
3,10,Counter-Strike,• Do you like original games? • Do you like ga...,1,0
4,10,Counter-Strike,"Easy to learn, hard to master.",1,1


In [3]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

{'Audio Production', 'RPG', 'Casual', 'Violent', 'Sports', 'Animation & Modeling', 'Free To Play', 'Strategy', 'Utilities', 'Tutorial', 'Movie', 'Design & Illustration', 'Video Production', 'Education', 'Short', '360 Video', 'nan', 'Action', 'Adventure', 'Simulation', 'Episodic', 'Nudity', 'Early Access', 'Accounting', 'Indie', 'Racing', 'Web Publishing', 'Gore', 'Documentary', 'Sexual Content', 'Game Development', 'Photo Editing', 'Massively Multiplayer', 'Software Training'}
34
{'Steam Machine', 'Great Soundtrack', 'RPG', 'Puzzle-Platformer', 'Boomer Shooter', 'Violent', 'Score Attack', 'Skiing', 'Logic', 'Idler', 'Vampire', 'Football', 'Management', 'Agriculture', 'Games Workshop', 'Episodic', 'Tennis', 'Point & Click', '3D Platformer', 'Dating Sim', 'FMV', 'Vehicular Combat', 'Dystopian', 'Competitive', 'Survival Horror', 'Historical', 'Soundtrack', 'Fishing', '2D', 'Gothic', 'Souls-like', 'Resource Management', 'Warhammer 40K', 'Local Co-Op', 'Bikes', 'Casual', 'Western', 'Action 

In [4]:

narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
1108640,OMNIMUS,"Sep 25, 2019",0 - 20000,0,0,0.99,0,1,Omnimus is an interactive visual novel in an u...,"['English', 'Russian']",...,0,0,0,Angry games,Total,"Single-player,Family Sharing","Adventure,Indie","Adventure,Indie,Visual Novel,Interactive Ficti...",https://shared.akamai.steamstatic.com/store_it...,NaN
1514380,COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣,"Jan 26, 2022",20000 - 50000,0,0,12.49,0,3,■Defy Death by Rewinding Time! Any hit you tak...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,"Gemdrops, Inc.","Gemdrops, Inc.","Single-player,Steam Achievements,Full controll...",Action,"Action-Adventure,Pixel Graphics,Time Manipulat...",https://shared.akamai.steamstatic.com/store_it...,NaN
1479370,Pixel Game Maker Series Werewolf Princess Kaguya,"Dec 20, 2020",0 - 20000,0,0,9.99,0,0,A stylish &amp; exciting action game! Journey ...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,SERIALGAMES Inc.,Gotcha Gotcha Games,"Single-player,Full controller support,Family S...","Action,Indie","Action,Arcade,Character Action Game,2D Platfor...",https://shared.akamai.steamstatic.com/store_it...,NaN
3098390,Spookstopia: Atomic Division,"Jul 31, 2024",0 - 20000,0,0,2.99,0,0,What is possible between the dimensions of Hel...,"['English', 'Russian']",...,0,0,0,TexFrendza,TexFrendza,"Single-player,Family Sharing","Indie,Early Access","Adventure,Platformer,2D Platformer,2D,Mystery,...",https://shared.akamai.steamstatic.com/store_it...,NaN
3215680,Who is Paul,"Oct 21, 2024",0 - 20000,0,0,0.89,70,0,WARNING: The game's content includes sudden lo...,"['English', 'French', 'Italian', 'German', 'Sp...",...,0,0,0,Atlas Sf,Atlas Sf,"Single-player,Steam Achievements,Family Sharing","Action,Adventure,Indie","Action,Adventure,FPS,Dark,Story Rich,Stealth,A...",https://shared.akamai.steamstatic.com/store_it...,NaN


In [5]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

filtered_reviews

,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,"First of all, this is no game but a virtual in...",1,1
46178,104600,Portal 2 - The Final Hours,The Final Hours is not a game (though can't yo...,1,1
46179,104600,Portal 2 - The Final Hours,Get this if you are a geek about how games are...,1,1
46180,104600,Portal 2 - The Final Hours,IT'S NOT A GAME! But it's a very interesting r...,1,1
46181,104600,Portal 2 - The Final Hours,Everything you would want to know about Portal...,1,1
...,...,...,...,...,...
6404109,9880,Champions Online,the gane starts to get boreing when you reatch...,-1,0
6404110,9880,Champions Online,Best character creator i've seen. EDIT: They ...,-1,0
6404111,9880,Champions Online,Region banned!! why did u show me this game to...,-1,0
6404112,9880,Champions Online,why the ♥♥♥♥,-1,0


## Data Cleaning

In [6]:
separators = ['?', '!', '.']

pattern = '|'.join(map(re.escape, separators))

separated_reviews = []

for review in filtered_reviews["review_text"]:
    separated_reviews += (re.split(pattern, str(review)))

separated_reviews

['First of all, this is no game but a virtual interactive book, and a very good one as a matter of fact',
 " To everyone who is interested in the behind the scenes of Portal 2, and how a development of a game works at Valve, you're in for one hell of a ride",
 ' I absolutely recommend it, it shows the everyday routine of an employee at Valve, some very enlightening points of view on their known delays and some videos plus slideshows too keep you interested while reading',
 " If you read it all calmly and with some attention it will take you a bit less than an hour to go through it all, but I assure you, it's absolutely worth it",
 '',
 "The Final Hours is not a game (though can't you imagine one - set perhaps during the Seven Hour War, trying to escape as the facility collapses and the Combine invade the test chambers",
 "), it's a book",
 ' It covers how Portal was created, the complex gestation of the sequel, and the personalities involved at every step',
 ' The book is interesting, 

In [7]:
#analyze review by sentence for topic modeling
cleaned_reviews_separated = []

for review in separated_reviews:
    ascii = re.sub(r'[^\x00-\x7f]', r'', str(review)) #filters for ascii characters
    new_lines = re.sub(r'[\n]',' ', ascii) #removes new lines
    punctuation = re.sub(r'[!?.,()*/"1234567890+]',' ',new_lines) #removes certain characters used in reviews
    dashes = re.sub(r'- | -',' ',punctuation) #remove dashes not used to hyphenate words
    cleaned_reviews_separated.append(re.sub(r'\s+',' ',dashes)) #remove extra whitespace

cleaned_reviews_separated


#analyze reviews in total for senti analysis
cleaned_reviews = []

for review in filtered_reviews["review_text"]:
    ascii = re.sub(r'[^\x00-\x7f]', r'', str(review)) #filters for ascii characters
    new_lines = re.sub(r'[\n]',' ', ascii) #removes new lines
    punctuation = re.sub(r'[!?.,()*/"1234567890+]',' ',new_lines) #removes certain characters used in reviews
    dashes = re.sub(r'- | -',' ',punctuation) #remove dashes not used to hyphenate words
    cleaned_reviews.append(re.sub(r'\s+',' ',dashes)) #remove extra whitespace

cleaned_reviews



["First of all this is no game but a virtual interactive book and a very good one as a matter of fact To everyone who is interested in the behind the scenes of Portal and how a development of a game works at Valve you're in for one hell of a ride I absolutely recommend it it shows the everyday routine of an employee at Valve some very enlightening points of view on their known delays and some videos plus slideshows too keep you interested while reading If you read it all calmly and with some attention it will take you a bit less than an hour to go through it all but I assure you it's absolutely worth it ",
 "The Final Hours is not a game though can't you imagine one set perhaps during the Seven Hour War trying to escape as the facility collapses and the Combine invade the test chambers it's a book It covers how Portal was created the complex gestation of the sequel and the personalities involved at every step The book is interesting revealing a lot of behind the scenes info You learn w

In [8]:
lower_case_reviews = []
for review in cleaned_reviews:
    lower_case_reviews.append(review.lower())

lower_case_reviews

lower_case_reviews_separated = []
for review in cleaned_reviews_separated:
    lower_case_reviews_separated.append(review.lower())

lower_case_reviews_separated

['first of all this is no game but a virtual interactive book and a very good one as a matter of fact',
 " to everyone who is interested in the behind the scenes of portal and how a development of a game works at valve you're in for one hell of a ride",
 ' i absolutely recommend it it shows the everyday routine of an employee at valve some very enlightening points of view on their known delays and some videos plus slideshows too keep you interested while reading',
 " if you read it all calmly and with some attention it will take you a bit less than an hour to go through it all but i assure you it's absolutely worth it",
 '',
 "the final hours is not a game though can't you imagine one set perhaps during the seven hour war trying to escape as the facility collapses and the combine invade the test chambers",
 " it's a book",
 ' it covers how portal was created the complex gestation of the sequel and the personalities involved at every step',
 ' the book is interesting revealing a lot of 

In [9]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

word_tokens_separated = [ review.split() for review in lower_case_reviews_separated ]
word_tokens = [ review.split() for review in lower_case_reviews]

stop_words = set(stopwords.words('english'))
stop_words.add('us')


lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

lemmatized_reviews

['first game virtual interactive book good one matter fact everyone interested behind scene portal development game work valve one hell ride absolutely recommend show everyday routine employee valve enlightening point view known delay video plus slideshows keep interested reading read calmly attention take bit less hour go assure absolutely worth',
 "final hour not game though can't imagine one set perhaps seven hour war trying escape facility collapse combine invade test chamber book cover portal created complex gestation sequel personality involved every step book interesting revealing lot behind scene info learn complicated business creating computer game especially valve one something stress involved even company like valve creating game hard work take toll book make use computer nature embedding video interactive content not add much overall though picture word matter",
 'get geek game made',
 'not game interesting reading interested creative process behind portal everything valve

In [10]:
lemmatized_reviews_separated = []
for tokens in word_tokens_separated:
    lemmatized_reviews_separated.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews_separated = [" ".join(review) for review in lemmatized_reviews_separated]

lemmatized_reviews_separated

['first game virtual interactive book good one matter fact',
 'everyone interested behind scene portal development game work valve one hell ride',
 'absolutely recommend show everyday routine employee valve enlightening point view known delay video plus slideshows keep interested reading',
 'read calmly attention take bit less hour go assure absolutely worth',
 '',
 "final hour not game though can't imagine one set perhaps seven hour war trying escape facility collapse combine invade test chamber",
 'book',
 'cover portal created complex gestation sequel personality involved every step',
 'book interesting revealing lot behind scene info',
 'learn complicated business creating computer game especially valve one something stress involved',
 'even company like valve creating game hard work take toll',
 'book make use computer nature embedding video interactive content',
 'not add much overall though picture word matter',
 '',
 'get geek game made',
 '',
 'not game',
 'interesting reading